## Steps

1. Loading data  
2. EDA  
3. Data Preprocessing  
        3.1 Cleaning  
        3.2 Stemming  
        3.3 All together  
        3.4 Target Encoding  
4. Tokens Visualization  
5. Vectorization  
        5.1 Tuning CountVectorizer  
        5.2 TF‑IDF  
        5.3 Word Embedding: GloVe  
6. Modelling  
        6.1 Naive Bayes (Document-Term Matrix)  
        6.2 Naive Bayes (TF‑IDF)  
        6.3 XGBoost  
7. LSTM  
8. BERT  
9. NLP: Disaster tweets  
        9.1 EDA


In [1]:
pip install nbformat>=4.2.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import string
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter
import nltk
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from PIL import Image

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import os
import spacy
import random
from spacy.util import compounding
from spacy.util import minibatch
from collections import defaultdict
from collections import Counter
# import keras
# from keras.models import Sequential
# from keras.initializers import Constant
# from keras.layers import (LSTM,
#                           Embedding,
#                           BatchNormalization,
#                           Dense,
#                           TimeDistributed,
#                           Dropout,
#                           Bidirectional,
#                           Flatten,
#                           GlobalMaxPool1D)
# from keras.preprocessing.text import Tokenizer
# from keras.preprocessing.sequence import pad_sequences
# from keras.layers.embeddings import Embedding
# from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
# from keras.optimizers import Adam

# from sklearn.metrics import (
#     precision_score,
#     recall_score,
#     f1_score,
#     classification_report,
#     accuracy_score
# )

In [3]:
primary_blue="#496595"
primary_blue2="#85a1c1"
primary_blue3="#3f4d63"
primary_grey="#c6ccd8"
primary_black="#202022"
primary_bgcolor="f4f0ea"

primary_green=px.colors.qualitative.Plotly[2]

In [4]:
df = pd.read_csv("D:\SummerPEP\pep_genai\data\spam.csv", encoding='latin-1')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_15724\84810637.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("D:\SummerPEP\pep_genai\data\spam.csv", encoding='latin-1')


In [5]:
df.head

<bound method NDFrame.head of         v1                                                 v2 Unnamed: 2  \
0      ham  Go until jurong point, crazy.. Available only ...        NaN   
1      ham                      Ok lar... Joking wif u oni...        NaN   
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3      ham  U dun say so early hor... U c already then say...        NaN   
4      ham  Nah I don't think he goes to usf, he lives aro...        NaN   
...    ...                                                ...        ...   
5567  spam  This is the 2nd time we have tried 2 contact u...        NaN   
5568   ham              Will Ì_ b going to esplanade fr home?        NaN   
5569   ham  Pity, * was in mood for that. So...any other s...        NaN   
5570   ham  The guy did some bitching but I acted like i'd...        NaN   
5571   ham                         Rofl. Its true to its name        NaN   

     Unnamed: 3 Unnamed: 4  
0           NaN        NaN  

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 677.1 KB


In [7]:
df.describe()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [8]:
len(df.columns)

5

In [9]:
df = df.dropna(how='any', axis=1)
df.columns=['target','message']

In [10]:
df.head()

,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
df['message_len'] = df['message'].apply(lambda x: len(x.split(' ')))
df.head()

,target,message,message_len
0,ham,"Go until jurong point, crazy.. Available only ...",20
1,ham,Ok lar... Joking wif u oni...,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28
3,ham,U dun say so early hor... U c already then say...,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",13


In [12]:
print("Max Length of message: ", max(df['message_len']))
print("Min Length of message: ", min(df['message_len']))

Max Length of message:  171
Min Length of message:  1


In [13]:
df['message'].apply(lambda x: len(x.split('. ')))

0       3
1       2
2       2
3       2
4       1
       ..
5567    4
5568    1
5569    2
5570    1
5571    2
Name: message, Length: 5572, dtype: int64

In [14]:
df['target'].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [15]:
balance_counts=pd.DataFrame({'target': df['target'].value_counts().index, 'count': df['target'].value_counts().values})

In [16]:
balance_counts

,target,count
0,ham,4825
1,spam,747


In [17]:
print(balance_counts['count'][0])

4825


In [18]:
fig=go.Figure()

fig.add_trace(go.Bar(
    x=['ham'],
    y=[balance_counts['count'][0]],
    name='ham',
    text=[balance_counts['count'][0]],
    marker_color=primary_blue
))

fig.add_trace(go.Bar(
    x=['spam'],
    y=[balance_counts['count'][1]],
    name='spam',
    text=[balance_counts['count'][1]],
    marker_color=primary_green
))

fig.update_layout(
    title="<span style='font-size: 20px;'>Spam vs Ham Count</span>",
    xaxis_title="<span style='font-size: 16px;'>Target</span>",
    yaxis_title="<span style='font-size: 16px;'>Count</span>",
    plot_bgcolor='#f4f0ea',   # Fixed: added '#' prefix
    paper_bgcolor='#f4f0ea',  # Make sure this has '#' too if you used it
)

fig.show()

In [19]:
ham_df=df[df['target']=='ham']['message_len'].value_counts().sort_index()
spam_df=df[df['target']=='spam']['message_len'].value_counts().sort_index()

fig=go.Figure()

fig.add_trace(go.Scatter(
    x=ham_df.index,
    y=ham_df.values,
    name='ham',
    fill='tozeroy',
    marker_color=primary_blue
))

fig.add_trace(go.Scatter(
    x=spam_df.index,
    y=spam_df.values,
    name='spam',
    fill='tozeroy',
    marker_color=primary_grey
))

fig.update_layout(
    title="<span style='font-size: 20px;'>Spam vs Ham Count</span>",
    xaxis_title="<span style='font-size: 16px;'>Target</span>",
    yaxis_title="<span style='font-size: 16px;'>Count</span>",
    plot_bgcolor='#f4f0ea',   # Fixed: added '#' prefix
    paper_bgcolor='#f4f0ea',  # Make sure this has '#' too if you used it
)
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(range=[0, max(ham_df.max(), spam_df.max())])

fig.show()

In [ ]:
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:10: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:10: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_15724\511036377.py:5: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_15724\511036377.py:6: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_15724\511036377.py:10: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [21]:
df['message_clean']=df['message'].apply(clean_text)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry in a wkly comp to win fa cup final...
3,ham,U dun say so early hor... U c already then say...,11,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah i dont think he goes to usf he lives aroun...


In [22]:
import nltk
nltk.download('stopwords')

stop_words=stopwords.words('english')
more_stopwords=['u','im','c']
stop_words=stop_words+more_stopwords

[nltk_data] Downloading package stopwords to C:\Users\Satish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
def remove_stopwords(text):
    text=' '.join(word for word in text.split(' ') if word not in stop_words)
    return text

In [24]:
df['message_clean']=df['message_clean'].apply(remove_stopwords)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry wkly comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say early hor already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goes usf lives around though


In [25]:
stemmer = nltk.SnowballStemmer("english")

def stemm_text(text):
    text = ' '.join(stemmer.stem(word) for word in text.split(' '))
    return text

In [26]:
df['message_clean'] = df['message_clean'].apply(stemm_text)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entri wkli comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say earli hor alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goe usf live around though


In [27]:
def preprocess_data(text):
    # Clean puntuation, urls, and so on
    text = clean_text(text)
    # Remove stopwords
    text = ' '.join(word for word in text.split(' ') if word not in stop_words)
    # Stemm all the words in the sentence
    text = ' '.join(stemmer.stem(word) for word in text.split(' '))

    return text

In [28]:
df['message_clean'] = df['message_clean'].apply(preprocess_data)
df.head()

,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entri wkli comp win fa cup final tkts m...
3,ham,U dun say so early hor... U c already then say...,11,dun say ear hor alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah dont think goe usf live around though
